# Reproduce the README headline numbers

This notebook verifies the README's ablation matrix. After the restructure,
the headline is **+0.0183 IC, t = +4.34, Sharpe = +0.86 over 905 days
post-October-2022** on the 617-name PIT universe with 22 features. The
journey is presented as three orthogonal ablations isolating universe / PIT,
feature set, and regime.

| Check | Experiment ID | Expected IC | Expected Sharpe | What it tests |
|---|---|---|---|---|
| Chronological Stage 1 (13 features, subset, no PIT) | `extended_kaggle_v2` | **+0.0075** | +0.28 | Historical baseline (weak features) |
| Chronological Stage 2 (13 features, PIT) | `extended_kaggle_v2_pit` | **+0.0008** | -0.04 | 89% collapse on weak feature set |
| Chronological Stage 3 (16 features, PIT) | `extended_kaggle_v2_anomaly` | **+0.0033** | +0.083 | + 3 academic anomalies |
| Headline baseline (22 features, PIT, full sample) | `extended_kaggle_v2_ohlcv` | **+0.0055** | +0.21 | + 6 OHLCV/volume |
| PIT-off ablation (22 features, subset, full sample) | `extended_kaggle_v2_ohlcv_subset` | **+0.0142** | +0.39 | Survivorship cost on strong features |
| **Headline regime split (22 features, PIT, post-Oct-2022)** | (subset of v2_ohlcv predictions) | **+0.0183** | **+0.86** | The headline number |

## How to use this notebook

1. Run all six CLI experiments first (see README "Reproduce the ablations").
   Each one writes predictions to the DuckDB store under
   `artifacts/predictions/predictions.duckdb`.
2. Run all cells of this notebook in order.
3. The final cell prints **PASS / FAIL per check**. Small drift (within ±5%
   of the quoted IC) is normal and counted as PASS.

## What this notebook does NOT do

- It does **not run the experiments for you.** Better to run them from the
  terminal and use this notebook to verify the stored predictions.
- It does **not refetch market data.** Reads from the cached panel + the
  prediction store. If you haven't done `cli refresh-data`, the joins return
  empty.
- It does **not check transaction-cost-adjusted Sharpe.** All numbers are gross.

## Prerequisites checklist

Before running this notebook, you should have completed all of:

```bash
pip install -e ".[dev,classical]"
python -m price_model.cli build-universe --name sp500_pit --start 2017-01-01
python -m price_model.cli refresh-data --universe sp500   --start 2017-01-01
python -m price_model.cli refresh-data --universe sp500_pit --start 2017-01-01
python -m price_model.cli run -e extended_kaggle_v2
python -m price_model.cli run -e extended_kaggle_v2_pit
python -m price_model.cli run -e extended_kaggle_v2_anomaly
python -m price_model.cli run -e extended_kaggle_v2_ohlcv
python -m price_model.cli run -e extended_kaggle_v2_ohlcv_subset
```

Cumulative wall-clock: ~90-120 minutes cold, ~15 minutes warm.


In [ ]:
from __future__ import annotations

from datetime import date

import polars as pl

from price_model.serving.store import PredictionStore
from price_model.data.loaders import load_panel
from price_model.features.targets import add_forward_excess_return
from price_model.eval.metrics import summarize
from price_model.eval.robustness import time_split_evaluate

# Expected README headlines, with tolerance bands.
EXPECTED = {
    "stage_1_naive": {
        "experiment":   "extended_kaggle_v2",
        "model_id":     "lightgbm_kaggle_v2",
        "universe":     "sp500",
        "pit_filter":   False,
        "ic_expected":  +0.0075,
        "ic_tolerance": 0.0040,
        "sharpe_expected": +0.28,
        "sharpe_tolerance": 0.20,
        "description": "13 features, subset, no PIT — chronological Stage 1 (historical)",
    },
    "stage_2_pit": {
        "experiment":   "extended_kaggle_v2_pit",
        "model_id":     "lightgbm_kaggle_v2_pit",
        "universe":     "sp500_pit",
        "pit_filter":   True,
        "ic_expected":  +0.0008,
        "ic_tolerance": 0.0020,
        "sharpe_expected": -0.04,
        "sharpe_tolerance": 0.15,
        "description": "13 features, PIT — chronological Stage 2 (89% collapse on weak features)",
    },
    "stage_3_anomaly": {
        "experiment":   "extended_kaggle_v2_anomaly",
        "model_id":     "lightgbm_kaggle_v2_anomaly",
        "universe":     "sp500_pit",
        "pit_filter":   True,
        "ic_expected":  +0.0033,
        "ic_tolerance": 0.0025,
        "sharpe_expected": +0.083,
        "sharpe_tolerance": 0.20,
        "description": "16 features (+ academic anomalies), PIT — chronological Stage 3",
    },
    "headline_baseline": {
        "experiment":   "extended_kaggle_v2_ohlcv",
        "model_id":     "lightgbm_kaggle_v2_ohlcv",
        "universe":     "sp500_pit",
        "pit_filter":   True,
        "ic_expected":  +0.0055,
        "ic_tolerance": 0.0030,
        "sharpe_expected": +0.21,
        "sharpe_tolerance": 0.15,
        "description": "22 features (headline model), PIT, full sample",
    },
    "pit_off_ablation": {
        "experiment":   "extended_kaggle_v2_ohlcv_subset",
        "model_id":     "lightgbm_kaggle_v2_ohlcv_subset",
        "universe":     "sp500",
        "pit_filter":   False,
        "ic_expected":  +0.0142,
        "ic_tolerance": 0.0040,
        "sharpe_expected": +0.39,
        "sharpe_tolerance": 0.20,
        "description": "22 features, subset / no PIT — survivorship-bias cost on the strong feature set",
    },
    "headline_regime": {
        "experiment":   "extended_kaggle_v2_ohlcv",   # same predictions, just sliced
        "model_id":     "lightgbm_kaggle_v2_ohlcv",
        "universe":     "sp500_pit",
        "pit_filter":   True,
        "cutoff":       date(2022, 10, 10),
        "ic_expected":  +0.0183,
        "ic_tolerance": 0.0050,
        "sharpe_expected": +0.86,
        "sharpe_tolerance": 0.25,
        "description": "22 features, PIT, post-October-2022 — THE HEADLINE",
    },

}

print("Expected README values:")
for stage, cfg in EXPECTED.items():
    print(f"  {stage:20s} IC = {cfg['ic_expected']:+.4f}   ({cfg['description']})")

## Helper functions

Two utilities:
- `fetch_predictions(model_id)` — pull the latest predictions for a model from the store, deduping on `generated_at` per `(date, ticker)`.
- `join_to_realized(preds, universe, pit_filter)` — join to the realized 5-day forward excess return panel.

In [2]:
def fetch_predictions(model_id: str) -> pl.DataFrame:
    """Pull predictions for one model_id, deduplicated to the latest generated_at."""
    store = PredictionStore(read_only=True)
    try:
        df = store.query(f"""
            WITH dedup AS (
                SELECT model_id, prediction_date, ticker, prediction,
                    ROW_NUMBER() OVER (
                        PARTITION BY model_id, prediction_date, ticker
                        ORDER BY generated_at DESC
                    ) AS rn
                FROM predictions
                WHERE model_id = '{model_id}'
            )
            SELECT prediction_date AS date, ticker, prediction
            FROM dedup
            WHERE rn = 1
        """)
    finally:
        store.close()
    return df


def join_to_realized(preds: pl.DataFrame, universe: str, pit_filter: bool) -> pl.DataFrame:
    """Join predictions to the realized 5-day forward excess return from the price panel."""
    panel = load_panel(universe=universe, start="2017-01-01", pit_filter=pit_filter)
    panel = add_forward_excess_return(panel, horizon_days=5)
    realized = panel.select("date", "ticker", pl.col("y").alias("realized"))
    return preds.join(realized, on=["date", "ticker"], how="inner")

## Verify each stage

The next four cells each run one stage's verification. Output format:

```
stage_1_naive          PASS   IC = +0.0073  (expected +0.0075, |drift| = 0.0002 < 0.0040)
```

or:

```
stage_1_naive          FAIL   no predictions found in store for model_id 'lightgbm_kaggle_v2'
                              -> did you run `python -m price_model.cli run -e extended_kaggle_v2` first?
```

In [ ]:
def verify(stage_name: str) -> dict[str, object]:
    """Verify a single stage. Returns a result dict for the final summary."""
    cfg = EXPECTED[stage_name]
    preds = fetch_predictions(cfg["model_id"])
    if preds.height == 0:
        return {
            "stage": stage_name,
            "status": "FAIL",
            "reason": (
                f"no predictions found in store for model_id '{cfg['model_id']}'\n"
                f"  -> did you run `python -m price_model.cli run -e {cfg['experiment']}` first?"
            ),
        }

    eval_df = join_to_realized(preds, cfg["universe"], cfg["pit_filter"])
    if eval_df.height == 0:
        return {
            "stage": stage_name,
            "status": "FAIL",
            "reason": f"prediction-realized join returned 0 rows for universe '{cfg['universe']}'",
        }

    # Stages 1-3: compute full-sample IC. Stage 4: compute post-cutoff IC + Sharpe.
    if stage_name == "stage_5_ohlcv_regime":
        split = time_split_evaluate(eval_df, cutoff=cfg["cutoff"], horizon_days=5)
        post = split.metrics_b
        ic = post.information_coefficient
        sharpe = post.long_short_sharpe
        n_dates = post.n_dates
    else:
        headline = summarize(eval_df, horizon_days=5)
        ic = headline.information_coefficient
        sharpe = headline.long_short_sharpe
        n_dates = headline.n_dates

    ic_drift = abs(ic - cfg["ic_expected"])
    ic_ok = ic_drift <= cfg["ic_tolerance"]
    sharpe_ok = True
    if "sharpe_expected" in cfg:
        sharpe_drift = abs(sharpe - cfg["sharpe_expected"])
        sharpe_ok = sharpe_drift <= cfg["sharpe_tolerance"]

    return {
        "stage": stage_name,
        "status": "PASS" if (ic_ok and sharpe_ok) else "DRIFT",
        "ic_actual": ic,
        "ic_expected": cfg["ic_expected"],
        "ic_drift": ic_drift,
        "ic_tolerance": cfg["ic_tolerance"],
        "sharpe_actual": sharpe,
        "sharpe_expected": cfg.get("sharpe_expected"),
        "n_dates": n_dates,
        "description": cfg["description"],
    }


def fmt_result(r: dict) -> str:
    if r["status"] == "FAIL":
        return f"{r['stage']:20s} FAIL   {r['reason']}"
    line = (
        f"{r['stage']:20s} {r['status']:5s}  "
        f"IC = {r['ic_actual']:+.4f}  (expected {r['ic_expected']:+.4f}, "
        f"|drift| = {r['ic_drift']:.4f} {'≤' if r['ic_drift'] <= r['ic_tolerance'] else '>'}"
        f" {r['ic_tolerance']:.4f})"
    )
    if r.get("sharpe_expected") is not None:
        line += f"\n{'':20s}        Sharpe = {r['sharpe_actual']:+.3f}  (expected {r['sharpe_expected']:+.3f})"
    line += f"\n{'':20s}        n_dates = {r['n_dates']:,}  ({r['description']})"
    return line

In [ ]:
# Verify all four stages
results = []
for stage_name in EXPECTED:
    r = verify(stage_name)
    results.append(r)
    print(fmt_result(r))
    print()

## Final summary

If all four stages show `PASS`, your reproduction matches the README headlines within reasonable yfinance drift.

If one or more stages show `DRIFT`, the prediction was generated but the numerical value moved more than the tolerance band. Most common causes:
- Your yfinance cache is significantly newer than the snapshot used to write the README (Wikipedia membership has changed; new tickers were added/removed).
- Your Ken French factor file has a different cutoff date (KF refreshes monthly).
- Hyperparameter drift if the YAML was edited.

If one or more stages show `FAIL`, the relevant experiment hasn't been run yet. Re-read the prerequisites checklist at the top.

In [ ]:
status_counts = {"PASS": 0, "DRIFT": 0, "FAIL": 0}
for r in results:
    status_counts[r["status"]] += 1

print(f"Summary: {status_counts['PASS']} PASS, {status_counts['DRIFT']} DRIFT, {status_counts['FAIL']} FAIL")
if status_counts["FAIL"] > 0:
    print("\n→ At least one experiment is missing from the store. Run the CLI commands in the prerequisites checklist.")
elif status_counts["DRIFT"] > 0:
    print("\n→ All experiments produced predictions, but at least one IC drifted beyond tolerance.")
    print("  Check yfinance data freshness or KF refresh date relative to the README snapshot.")
else:
    print("\n→ All six ablation cells reproduce within tolerance. The headline of")
    print("  +0.0183 IC / Sharpe +0.86 (post-October-2022, 22 features, PIT) is independently")
    print("  verified, and the three orthogonal ablations (universe/PIT, features, regime)")
    print("  reproduce on your data.")

## Honest caveats about reproduction

Even a perfect PASS on all four stages doesn't mean the underlying findings would survive every test:

- The four headline numbers are computed on the same data window and ticker universe described in the README. Sliding the start date by a year or running on a different country's index would produce different numbers; we haven't tested those.
- The +0.0116 post-October-2022 IC is gross of transaction costs, taxes, and slippage. After realistic retail costs, the after-cost edge for an individual investor is approximately zero.
- yfinance's drop-list (~12% of historical S&P 500 members, including SIVB / FRC / SBNY) means our PIT correction is partial. A truly bias-free backtest with paid data would likely produce somewhat lower post-2022 IC because the bank failures would be in the cross-section.

See the README's "Data quality and methodological limitations" section for the full enumeration.